# 🎯 Arabic CV Analyzer — v3
## AraBERT vs CAMeL-BERT | كل التحسينات

| التحسين | التفاصيل |
|---------|----------|
| ✅ Data-driven Thresholds | من percentiles الداتا |
| ✅ HuberMSE Loss | أحسن من MSE مع الـ outliers |
| ✅ Freeze → Unfreeze | استقرار في البداية |
| ✅ Layer-wise LR Decay | lr مختلفة لكل طبقة BERT |
| ✅ Ensemble Prediction | بيجمع الموديلين |
| ✅ Linear Warmup + Clipping | من v2 |

---
## 1️⃣ Imports & Config

In [ ]:
# !pip install transformers torch scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
import re, warnings, json, pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from IPython.display import display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score,
    mean_absolute_error, mean_squared_error, r2_score
)
from scipy.stats import spearmanr

warnings.filterwarnings('ignore')

# ── Style ─────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#f8f9fa',
    'axes.grid'        : True,
    'grid.alpha'       : 0.4,
    'axes.spines.top'  : False,
    'axes.spines.right': False,
    'font.size'        : 11,
    'axes.titlesize'   : 13,
    'axes.titleweight' : 'bold',
})
CLRS = {'AraBERT': '#2563EB', 'CAMeL-BERT': '#EA580C', 'Ensemble': '#7C3AED'}

device   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SAVE_DIR = Path('saved_model')
SAVE_DIR.mkdir(exist_ok=True)

print(f'🖥️  Device : {device}')
print(f'🔦  PyTorch: {torch.__version__}')

MODELS = {
    'AraBERT'   : 'aubmindlab/bert-base-arabertv2',
    'CAMeL-BERT': 'CAMeL-Lab/bert-base-arabic-camelbert-mix'
}
CLASS_NAMES  = ['ضعيف', 'متوسط', 'جيد', 'ممتاز']
CLASS_LABELS = [0, 1, 2, 3]

BEST_PARAMS = {
    'lr'            : 1e-5,
    'head_lr'       : 1e-4,   # lr أعلى للـ head والـ attention pooling
    'batch_size'    : 8,
    'dropout'       : 0.2,
    'max_grad_norm' : 1.0,
    'warmup_ratio'  : 0.1,
    'weight_decay'  : 0.01,
    'lr_decay'      : 0.9,    # Layer-wise LR decay factor
    'freeze_epochs' : 2,      # عدد الـ epochs اللي الـ BERT فيها مفروز
}
FULL_EPOCHS = 10
PATIENCE    = 3

print(f'\n📌 Hyperparams:')
for k, v in BEST_PARAMS.items():
    print(f'   {k:<16} = {v}')

---
## 2️⃣ Data Loading & Exploration

In [ ]:
df = pd.read_csv('arabic_cvs_with_scores.csv', encoding='utf-8-sig')
df['cv_words'] = df['arabic_cv'].astype(str).apply(lambda x: len(x.split()))
print(f'✅ {len(df):,} CVs  |  Columns: {list(df.columns)}')
display(df.head(3))

In [ ]:
fig = plt.figure(figsize=(18, 11))
fig.suptitle('📊 Data Exploration', fontsize=16, fontweight='bold', y=1.01)

ax1 = fig.add_subplot(2, 3, 1)
cc  = df['category'].value_counts()
bars = ax1.barh(cc.index[:12], cc.values[:12],
                color=plt.cm.Blues_r(np.linspace(0.3, 0.9, len(cc[:12]))))
ax1.set_xlabel('عدد الـ CVs'); ax1.set_title('توزيع CVs حسب الفئة')
for bar, v in zip(bars, cc.values[:12]):
    ax1.text(v+.3, bar.get_y()+bar.get_height()/2, str(v), va='center', fontsize=9)

ax2 = fig.add_subplot(2, 3, 2)
ax2.hist(df['suitability_score'].dropna(), bins=25, color='#2563EB', edgecolor='white', alpha=0.85)
m = df['suitability_score'].mean()
ax2.axvline(m, color='red', linestyle='--', lw=1.5, label=f'Mean={m:.1f}')
ax2.set_xlabel('Score'); ax2.set_ylabel('Count')
ax2.set_title('توزيع درجات الملاءمة'); ax2.legend()

ax3 = fig.add_subplot(2, 3, 3)
ax3.hist(df['cv_words'], bins=30, color='#059669', edgecolor='white', alpha=0.85)
m2 = df['cv_words'].mean()
ax3.axvline(m2, color='red', linestyle='--', lw=1.5, label=f'Mean={m2:.0f}')
ax3.set_xlabel('عدد الكلمات'); ax3.set_title('توزيع طول الـ CV'); ax3.legend()

ax4 = fig.add_subplot(2, 3, 4)
cs   = df.groupby('category')['suitability_score'].mean().sort_values()
cols = ['#EF4444' if v<60 else '#22C55E' for v in cs.values]
ax4.barh(cs.index, cs.values, color=cols)
ax4.axvline(60, color='gray', linestyle=':', alpha=.7)
ax4.set_xlabel('Avg Score'); ax4.set_title('متوسط الدرجة حسب الفئة')
for i, v in enumerate(cs.values):
    ax4.text(v+.3, i, f'{v:.1f}', va='center', fontsize=8)

ax5 = fig.add_subplot(2, 3, 5)
cats     = df.groupby('category')['suitability_score'].median().sort_values().index
data_box = [df[df['category']==c]['suitability_score'].dropna().values for c in cats]
ax5.boxplot(data_box, vert=False, patch_artist=True,
            boxprops=dict(facecolor='#BFDBFE', color='#1E40AF'),
            medianprops=dict(color='#DC2626', linewidth=2))
ax5.set_yticks(range(1, len(cats)+1)); ax5.set_yticklabels(cats, fontsize=8)
ax5.set_title('توزيع الدرجات (Boxplot)')

ax6 = fig.add_subplot(2, 3, 6)
sc  = ax6.scatter(df['cv_words'], df['suitability_score'],
                  alpha=.4, s=12, c=df['suitability_score'], cmap='RdYlGn', vmin=0, vmax=100)
plt.colorbar(sc, ax=ax6, label='Score')
ax6.set_xlabel('عدد الكلمات'); ax6.set_ylabel('Score')
ax6.set_title('طول الـ CV vs الدرجة')

plt.tight_layout()
plt.savefig('01_exploration.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3️⃣ ATS Engine + Preprocessing

In [ ]:
DOMAIN_KEYWORDS = {
    'تكنولوجيا المعلومات': ['python','java','javascript','sql','nosql','react','node','docker','kubernetes','aws','azure','git','api','rest','machine learning','deep learning','linux','agile','scrum','ci/cd'],
    'موارد بشرية'        : ['توظيف','تدريب','تطوير','أداء','رواتب','تعويضات','علاقات عمل','kpi','okr','onboarding','hris','payroll','retention','culture','succession','talent','engagement'],
    'مالية ومحاسبة'      : ['excel','sap','oracle','ifrs','gaap','ميزانية','تدقيق','ضرائب','تحليل مالي','تقارير','quickbooks','erp','cpa','cma','dcf'],
    'هندسة'              : ['autocad','solidworks','matlab','ansys','pmp','iso','six sigma','lean','revit','primavera','bim','حديد','خرسانة','ميكانيكا','كهرباء','plc','scada'],
    'تسويق'              : ['seo','sem','google ads','facebook ads','content','analytics','crm','hubspot','mailchimp','brand','roi','kpi','digital','social media','copywriting','strategy'],
    'مبيعات'             : ['مبيعات','عملاء','crm','salesforce','target','pipeline','negotiation','b2b','b2c','revenue','quota','cold calling','account management','upsell'],
    'رعاية صحية'         : ['سريري','مرضى','تشخيص','علاج','أدوية','مختبر','تمريض','طوارئ','جراحة','emr','his','bls','acls','hipaa'],
    'تعليم'              : ['مناهج','تدريس','طلاب','تقييم','تعليم إلكتروني','lms','moodle','خطة درس','أهداف تعليمية','bloom','differentiation'],
    'إعلام وتصميم'       : ['photoshop','illustrator','figma','adobe','premiere','after effects','typography','branding','ui','ux','indesign','3d','animation'],
    'سياحة ومطاعم'       : ['hospitality','opera','خدمة عملاء','haccp','food safety','فنادق','مطاعم','حجوزات','pos','revenue management'],
}
DEFAULT_KEYWORDS = ['خبرة','مهارات','تواصل','قيادة','تحليل','مشاريع','فريق','إدارة','تطوير','تحسين','excel','word','powerpoint']
ATS_MESSAGES = {
    'has_experience': 'أضف قسم خبرة العمل بالتفصيل',
    'has_education' : 'أضف قسم التعليم والمؤهلات',
    'has_skills'    : 'أضف قسم المهارات التقنية والشخصية',
    'has_summary'   : 'أضف ملخصاً مهنياً في بداية الـ CV',
    'has_email'     : 'أضف بريدك الإلكتروني',
    'has_phone'     : 'أضف رقم هاتفك',
    'good_length'   : 'اجعل الـ CV بين 200-800 كلمة',
    'has_contact'   : 'أضف رابط LinkedIn أو GitHub أو Portfolio'
}

def calculate_ats_score(arabic_cv, category=''):
    cv = str(arabic_cv).lower()
    checks = {
        'has_experience': any(k in cv for k in ['الخبرات','خبرة العمل','الخبرة المهنية']),
        'has_education' : any(k in cv for k in ['التعليم','المؤهلات','الدراسة']),
        'has_skills'    : any(k in cv for k in ['المهارات','الكفاءات','قدرات']),
        'has_summary'   : any(k in cv for k in ['ملخص','نبذة','عن نفسي','profile']),
        'has_email'     : bool(re.search(r'[\w\.-]+@[\w\.-]+', cv)),
        'has_phone'     : bool(re.search(r'[\+\d][\d\s\-]{8,}', cv)),
        'good_length'   : 200 <= len(cv.split()) <= 800,
        'has_contact'   : any(k in cv for k in ['linkedin','github','portfolio','موقع'])
    }
    w = {'has_experience':15,'has_education':12,'has_skills':12,'has_summary':8,
         'has_email':6,'has_phone':4,'good_length':6,'has_contact':7}
    struct_score  = sum(w[k] for k,v in checks.items() if v)
    kws           = DOMAIN_KEYWORDS.get(category, DEFAULT_KEYWORDS)
    matched       = [k for k in kws if k.lower() in cv]
    keyword_score = (len(matched)/max(len(kws),1))*30
    return min(int(struct_score+keyword_score),100), checks, matched

def clean_arabic_text(text):
    text = str(text)
    text = re.sub(r'[\u064B-\u065F]','',text)
    text = re.sub(r'[إأآا]','ا',text)
    text = re.sub(r'ة','ه',text)
    text = re.sub(r'ى','ي',text)
    text = re.sub(r'[^\w\s\u0600-\u06FF]',' ',text)
    return re.sub(r'\s+',' ',text).strip()

ats_r              = df.apply(lambda r: calculate_ats_score(r['arabic_cv'],r['category']),axis=1)
df['ats_score']    = ats_r.apply(lambda x: x[0])
df['ats_checks']   = ats_r.apply(lambda x: x[1])
df['ats_matched']  = ats_r.apply(lambda x: x[2])
df['ats_kw_count'] = df['ats_matched'].apply(len)
df['arabic_cv_clean'] = df['arabic_cv'].apply(clean_arabic_text)

# ── ✅ Input مُحسَّن: ATS bucket + category + CV ──────
def build_input(row):
    ats = row['ats_score']
    bucket = 'ats_low' if ats < 50 else ('ats_mid' if ats < 75 else 'ats_high')
    return f"{row['category']} {bucket} [SEP] {row['arabic_cv_clean']}"

df['input_text'] = df.apply(build_input, axis=1)
df_clean = df.dropna(subset=['arabic_cv_clean','suitability_score']).copy()

train_df, temp_df = train_test_split(df_clean, test_size=.2, random_state=42)
val_df,   test_df = train_test_split(temp_df,  test_size=.5, random_state=42)
print(f'✅ Train:{len(train_df):,}  Val:{len(val_df):,}  Test:{len(test_df):,}')

---
## 4️⃣ Score → Class | ✅ Data-driven Thresholds

In [ ]:
# ✅ Thresholds من percentiles الـ train data (مش ثابتة)
T1 = float(np.percentile(train_df['suitability_score'], 25))
T2 = float(np.percentile(train_df['suitability_score'], 50))
T3 = float(np.percentile(train_df['suitability_score'], 75))

print(f'📌 Data-driven Thresholds (P25/P50/P75):')
print(f'   ضعيف  < {T1:.1f}')
print(f'   متوسط < {T2:.1f}')
print(f'   جيد   < {T3:.1f}')
print(f'   ممتاز ≥ {T3:.1f}')

def score_to_class(score):
    if   score < T1: return 0
    elif score < T2: return 1
    elif score < T3: return 2
    else:            return 3

# Distribution check — لازم يكون متوازن تقريباً ~25% كل class
df_clean['label'] = df_clean['suitability_score'].apply(score_to_class)
lc = df_clean['label'].map(dict(enumerate(CLASS_NAMES))).value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('🎯 Class Distribution — Data-driven Thresholds', fontweight='bold')

colors = ['#EF4444','#F59E0B','#22C55E','#2563EB']
vals   = [lc.get(cn, 0) for cn in CLASS_NAMES]
bars   = axes[0].bar(CLASS_NAMES, vals, color=colors)
axes[0].set_ylabel('Count'); axes[0].set_title('Class Count')
for bar in bars:
    axes[0].text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                 f'{int(bar.get_height())}\n({int(bar.get_height())/len(df_clean)*100:.0f}%)',
                 ha='center', fontsize=9, fontweight='bold')

axes[1].hist(df_clean['suitability_score'], bins=30, color='#BFDBFE', edgecolor='white')
for t, lbl, clr in zip([T1,T2,T3], [f'T1={T1:.0f}',f'T2={T2:.0f}',f'T3={T3:.0f}'],
                        ['#EF4444','#F59E0B','#22C55E']):
    axes[1].axvline(t, color=clr, linestyle='--', lw=2, label=lbl)
axes[1].set_xlabel('Score'); axes[1].set_ylabel('Count')
axes[1].set_title('Score Distribution + Thresholds'); axes[1].legend()

plt.tight_layout(); plt.savefig('02_class_dist.png', dpi=150, bbox_inches='tight'); plt.show()

---
## 5️⃣ Model Architecture

In [ ]:
class CVDatasetSliding(Dataset):
    def __init__(self, df, tokenizer, max_len=512, stride=384):
        self.df=df.reset_index(drop=True); self.tokenizer=tokenizer
        self.max_len=max_len; self.stride=stride
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        toks = self.tokenizer(row['input_text'],truncation=True,max_length=512,add_special_tokens=True)
        ids  = toks['input_ids']
        chunks_ids, chunks_mask = [], []
        start = 0
        while start < len(ids):
            end=min(start+self.max_len,len(ids))
            chunk=torch.tensor(ids[start:end],dtype=torch.long)
            mask=torch.ones(len(chunk),dtype=torch.long)
            pad=self.max_len-len(chunk)
            if pad>0:
                chunk=torch.cat([chunk,torch.zeros(pad,dtype=torch.long)])
                mask=torch.cat([mask,torch.zeros(pad,dtype=torch.long)])
            chunks_ids.append(chunk); chunks_mask.append(mask)
            if end==len(ids): break
            start+=self.stride
        return {'chunks_ids':torch.stack(chunks_ids),'chunks_mask':torch.stack(chunks_mask),
                'score':torch.tensor(row['suitability_score']/100.,dtype=torch.float),
                'n_chunks':len(chunks_ids)}

def collate_fn(batch):
    max_c=max(b['n_chunks'] for b in batch)
    ids_l,mask_l,scores=[],[],[]
    for b in batch:
        p=max_c-b['n_chunks']
        if p>0:
            ids_l.append(torch.cat([b['chunks_ids'],torch.zeros(p,512,dtype=torch.long)]))
            mask_l.append(torch.cat([b['chunks_mask'],torch.zeros(p,512,dtype=torch.long)]))
        else:
            ids_l.append(b['chunks_ids']); mask_l.append(b['chunks_mask'])
        scores.append(b['score'])
    return {'chunks_ids':torch.stack(ids_l),'chunks_mask':torch.stack(mask_l),
            'n_chunks':[b['n_chunks'] for b in batch],'score':torch.stack(scores)}

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size=768):
        super().__init__()
        self.attention=nn.Sequential(nn.Linear(hidden_size,256),nn.Tanh(),nn.Linear(256,1))
    def forward(self, cls_embeddings, n_chunks):
        pooled=[]
        for i in range(cls_embeddings.shape[0]):
            n=n_chunks[i]; emb=cls_embeddings[i,:n,:]
            att=F.softmax(self.attention(emb),dim=0)
            pooled.append((att*emb).sum(dim=0))
        return torch.stack(pooled)

class CVRegressor(nn.Module):
    def __init__(self, model_path, dropout=0.2):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(model_path)
        self.att_pool = AttentionPooling(768)
        self.head     = nn.Sequential(
            nn.Linear(768,256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256,64),  nn.GELU(), nn.Dropout(dropout),
            nn.Linear(64,1),    nn.Sigmoid())
    def forward(self, chunks_ids, chunks_mask, n_chunks):
        B,C,L=chunks_ids.shape
        out=self.bert(input_ids=chunks_ids.view(B*C,L),
                      attention_mask=chunks_mask.view(B*C,L),output_attentions=False)
        cls=out.last_hidden_state[:,0,:].view(B,C,768)
        return self.head(self.att_pool(cls,n_chunks)).squeeze(-1)

# ✅ HuberMSE Loss — أحسن من MSE مع الـ outliers
class HuberMSELoss(nn.Module):
    def __init__(self, delta=0.1, alpha=0.7):
        """
        alpha * MSE + (1-alpha) * Huber
        MSE بيعاقب الأخطاء الكبيرة أكتر
        Huber أكثر مقاومة للـ outliers
        """
        super().__init__()
        self.mse   = nn.MSELoss()
        self.huber = nn.HuberLoss(delta=delta)
        self.alpha = alpha
    def forward(self, pred, target):
        return self.alpha*self.mse(pred,target) + (1-self.alpha)*self.huber(pred,target)

def run_epoch(model, loader, optimizer, criterion, scheduler=None, train=True, max_grad_norm=1.0):
    model.train() if train else model.eval()
    total_loss,preds_list,scores_list=0,[],[]
    ctx=torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for batch in loader:
            ids=batch['chunks_ids'].to(device); mask=batch['chunks_mask'].to(device)
            scores=batch['score'].to(device); n_chunks=batch['n_chunks']
            if train: optimizer.zero_grad()
            pred=model(ids,mask,n_chunks)
            loss=criterion(pred,scores)
            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(),max_grad_norm)
                optimizer.step()
                if scheduler: scheduler.step()
            total_loss+=loss.item()
            preds_list.extend((pred*100).detach().cpu().tolist())
            scores_list.extend((scores*100).cpu().tolist())
    mae=mean_absolute_error(scores_list,preds_list)
    rmse=np.sqrt(mean_squared_error(scores_list,preds_list))
    pred_cls=[score_to_class(p) for p in preds_list]
    true_cls=[score_to_class(s) for s in scores_list]
    acc=sum(p==t for p,t in zip(pred_cls,true_cls))/len(true_cls)
    return {'loss':total_loss/len(loader),'mae':mae,'rmse':rmse,'acc':acc,
            'pred_scores':preds_list,'true_scores':scores_list,
            'pred_classes':pred_cls,'true_classes':true_cls}

print('✅ Architecture ready')

---
## 6️⃣ Training — كل التحسينات مع بعض

In [ ]:
def build_optimizer(model, base_lr, head_lr, weight_decay, lr_decay):
    """
    ✅ Layer-wise LR Decay:
    - الطبقات الأعمق في الـ BERT → lr أصغر
    - الـ head و attention pooling → head_lr أعلى
    - مفيش weight decay على bias و LayerNorm
    """
    no_decay    = ['bias','LayerNorm.weight']
    num_layers  = len(model.bert.encoder.layer)   # 12 في الغالب
    opt_params  = []

    # Embeddings — أبطأ طبقة
    emb_lr = base_lr * (lr_decay ** num_layers)
    opt_params += [
        {'params': [p for n,p in model.bert.embeddings.named_parameters() if not any(nd in n for nd in no_decay)],
         'lr': emb_lr, 'weight_decay': weight_decay},
        {'params': [p for n,p in model.bert.embeddings.named_parameters() if     any(nd in n for nd in no_decay)],
         'lr': emb_lr, 'weight_decay': 0.0},
    ]

    # Encoder layers — كل طبقة لها lr خاصة
    for i, layer in enumerate(model.bert.encoder.layer):
        layer_lr = base_lr * (lr_decay ** (num_layers - i))
        opt_params += [
            {'params': [p for n,p in layer.named_parameters() if not any(nd in n for nd in no_decay)],
             'lr': layer_lr, 'weight_decay': weight_decay},
            {'params': [p for n,p in layer.named_parameters() if     any(nd in n for nd in no_decay)],
             'lr': layer_lr, 'weight_decay': 0.0},
        ]

    # Head + Attention Pooling — أعلى lr
    opt_params += [
        {'params': [p for n,p in model.head.named_parameters()     if not any(nd in n for nd in no_decay)],
         'lr': head_lr, 'weight_decay': weight_decay},
        {'params': [p for n,p in model.head.named_parameters()     if     any(nd in n for nd in no_decay)],
         'lr': head_lr, 'weight_decay': 0.0},
        {'params': model.att_pool.parameters(),
         'lr': head_lr, 'weight_decay': weight_decay},
    ]

    return torch.optim.AdamW(opt_params)


final_models  = {}
final_history = {}
tokenizers    = {}

for model_name, model_path in MODELS.items():
    print(f'\n{"="*60}\n🚀 Training: {model_name}\n{"="*60}')

    tok = AutoTokenizer.from_pretrained(model_path)
    tokenizers[model_name] = tok

    train_ds = CVDatasetSliding(train_df, tok)
    val_ds   = CVDatasetSliding(val_df,   tok)
    train_dl = DataLoader(train_ds, batch_size=BEST_PARAMS['batch_size'], shuffle=True,  collate_fn=collate_fn)
    val_dl   = DataLoader(val_ds,   batch_size=BEST_PARAMS['batch_size'], collate_fn=collate_fn)

    model     = CVRegressor(model_path, dropout=BEST_PARAMS['dropout']).to(device)
    criterion = HuberMSELoss()
    save_path = SAVE_DIR / f'{model_name.replace("-","_")}_best.pt'

    # ── Phase 1: Freeze BERT — تدريب الـ head بس ─────────
    freeze_eps = BEST_PARAMS['freeze_epochs']
    print(f'\n  🧊 Phase 1: Freeze BERT ({freeze_eps} epochs — head only)')
    for param in model.bert.parameters():
        param.requires_grad = False

    opt_frozen = torch.optim.AdamW(
        list(model.head.parameters()) + list(model.att_pool.parameters()),
        lr=BEST_PARAMS['head_lr'], weight_decay=BEST_PARAMS['weight_decay']
    )
    frozen_steps   = len(train_dl) * freeze_eps
    sched_frozen   = get_linear_schedule_with_warmup(opt_frozen, 0, frozen_steps)
    history        = []
    best_mae       = float('inf')

    for epoch in range(freeze_eps):
        tr = run_epoch(model, train_dl, opt_frozen, criterion, sched_frozen, train=True,
                       max_grad_norm=BEST_PARAMS['max_grad_norm'])
        vl = run_epoch(model, val_dl,   opt_frozen, criterion, train=False)
        history.append({'t_loss':tr['loss'],'v_loss':vl['loss'],
                         't_mae':tr['mae'],'v_mae':vl['mae'],
                         't_acc':tr['acc'],'v_acc':vl['acc'],
                         'lr':opt_frozen.param_groups[0]['lr'],'phase':'frozen'})
        print(f'  🧊 Ep{epoch+1:02d}  loss={vl["loss"]:.4f}  mae={vl["mae"]:.2f}  acc={vl["acc"]:.3f}')
        if vl['mae'] < best_mae:
            best_mae = vl['mae']
            torch.save({'model_state':model.state_dict(),'model_name':model_name,
                        'model_path':model_path,'params':BEST_PARAMS,
                        'thresholds':(T1,T2,T3),'best_epoch':epoch+1,'best_mae':best_mae}, save_path)

    # ── Phase 2: Unfreeze — تدريب كل حاجة ───────────────
    remaining_eps = FULL_EPOCHS - freeze_eps
    print(f'\n  🔥 Phase 2: Unfreeze all ({remaining_eps} epochs — layer-wise LR)')
    for param in model.bert.parameters():
        param.requires_grad = True

    optimizer      = build_optimizer(model,
                         base_lr=BEST_PARAMS['lr'],
                         head_lr=BEST_PARAMS['head_lr'],
                         weight_decay=BEST_PARAMS['weight_decay'],
                         lr_decay=BEST_PARAMS['lr_decay'])
    total_steps    = len(train_dl) * remaining_eps
    warmup_steps   = int(total_steps * BEST_PARAMS['warmup_ratio'])
    scheduler      = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    patience_ctr   = 0

    for epoch in range(remaining_eps):
        tr = run_epoch(model, train_dl, optimizer, criterion, scheduler, train=True,
                       max_grad_norm=BEST_PARAMS['max_grad_norm'])
        vl = run_epoch(model, val_dl, optimizer, criterion, train=False)
        cur_lr = optimizer.param_groups[-1]['lr']   # head lr
        history.append({'t_loss':tr['loss'],'v_loss':vl['loss'],
                         't_mae':tr['mae'],'v_mae':vl['mae'],
                         't_acc':tr['acc'],'v_acc':vl['acc'],
                         'lr':cur_lr,'phase':'unfrozen'})
        flag = '💾' if vl['mae'] < best_mae else '  '
        print(f'  {flag} 🔥 Ep{epoch+freeze_eps+1:02d}  '
              f'loss={vl["loss"]:.4f}  mae={vl["mae"]:.2f}  '
              f'acc={vl["acc"]:.3f}  lr={cur_lr:.2e}')
        if vl['mae'] < best_mae:
            best_mae=vl['mae']; patience_ctr=0
            torch.save({'model_state':model.state_dict(),'model_name':model_name,
                        'model_path':model_path,'params':BEST_PARAMS,
                        'thresholds':(T1,T2,T3),
                        'best_epoch':epoch+freeze_eps+1,'best_mae':best_mae}, save_path)
        else:
            patience_ctr+=1
            if patience_ctr >= PATIENCE:
                print(f'  ⏹️  Early stopping'); break

    print(f'  ✅ Best MAE={best_mae:.2f}')
    final_models[model_name]=model; final_history[model_name]=history

print('\n✅ Training complete!')

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('📈 Training Curves — AraBERT vs CAMeL-BERT', fontsize=14, fontweight='bold')

for (val_k, train_k, label, ax) in [
    ('v_loss','t_loss','Loss',        axes[0,0]),
    ('v_mae', 't_mae', 'MAE',         axes[0,1]),
    ('v_acc', 't_acc', 'Accuracy',    axes[1,0]),
    ('lr',    None,    'Learning Rate',axes[1,1]),
]:
    for mn, hist in final_history.items():
        x = range(1, len(hist)+1)
        ax.plot(x, [h[val_k] for h in hist], '-o', ms=5,
                label=f'{mn} val', color=CLRS[mn], lw=2)
        if train_k:
            ax.plot(x, [h[train_k] for h in hist], '--',
                    label=f'{mn} train', color=CLRS[mn], alpha=.4, lw=1.5)
    # خط فاصل بين frozen وunfrozen
    ax.axvline(BEST_PARAMS['freeze_epochs']+.5, color='gray',
               linestyle=':', lw=1.5, alpha=.7)
    ax.text(BEST_PARAMS['freeze_epochs']+.6, ax.get_ylim()[1]*0.95,
            'unfreeze→', fontsize=8, color='gray')
    ax.set_xlabel('Epoch'); ax.set_ylabel(label); ax.set_title(label)
    ax.legend(fontsize=8)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))

plt.tight_layout(); plt.savefig('03_training_curves.png', dpi=150, bbox_inches='tight'); plt.show()

---
## 7️⃣ Evaluation + ✅ Ensemble

In [ ]:
eval_results = {}

for model_name, model_path in MODELS.items():
    ckpt  = torch.load(SAVE_DIR/f'{model_name.replace("-","_")}_best.pt',
                       map_location=device, weights_only=False)
    model = CVRegressor(model_path, dropout=BEST_PARAMS['dropout']).to(device)
    model.load_state_dict(ckpt['model_state'])
    final_models[model_name] = model

    test_dl = DataLoader(CVDatasetSliding(test_df, tokenizers[model_name]),
                         batch_size=8, collate_fn=collate_fn)
    res = run_epoch(model, test_dl, None, HuberMSELoss(), train=False)
    res['r2']       = r2_score(res['true_scores'], res['pred_scores'])
    res['spearman'] = spearmanr(res['true_scores'], res['pred_scores'])[0]
    eval_results[model_name] = res
    print(f'📊 {model_name}  MAE={res["mae"]:.2f}  Acc={res["acc"]*100:.1f}%  (best_ep={ckpt["best_epoch"]})')

# ✅ Ensemble — بنحسب weighted average بناءً على MAE
def get_scores_from_loader(model, loader):
    model.eval()
    preds = []
    with torch.no_grad():
        for batch in loader:
            ids=batch['chunks_ids'].to(device); mask=batch['chunks_mask'].to(device)
            n_chunks=batch['n_chunks']
            pred=model(ids,mask,n_chunks)
            preds.extend((pred*100).cpu().tolist())
    return np.array(preds)

# الوزن عكسي مع الـ MAE — اللي MAE أقل ليه وزن أعلى
maes    = {mn: eval_results[mn]['mae'] for mn in MODELS}
inv_mae = {mn: 1/v for mn,v in maes.items()}
total   = sum(inv_mae.values())
weights = {mn: v/total for mn,v in inv_mae.items()}
print(f'\n🔀 Ensemble weights: ', {mn:round(w,3) for mn,w in weights.items()})

# نحسب predictions للـ ensemble
ens_preds  = np.zeros(len(test_df))
true_scores = None
for model_name in MODELS:
    test_dl = DataLoader(CVDatasetSliding(test_df, tokenizers[model_name]),
                         batch_size=8, collate_fn=collate_fn)
    scores  = get_scores_from_loader(final_models[model_name], test_dl)
    ens_preds += weights[model_name] * scores
    if true_scores is None:
        true_scores = eval_results[model_name]['true_scores']

ens_cls   = [score_to_class(p) for p in ens_preds]
true_cls  = [score_to_class(s) for s in true_scores]
ens_mae   = mean_absolute_error(true_scores, ens_preds)
ens_acc   = sum(p==t for p,t in zip(ens_cls,true_cls)) / len(true_cls)
ens_r2    = r2_score(true_scores, ens_preds)
ens_sp    = spearmanr(true_scores, ens_preds)[0]
eval_results['Ensemble'] = {
    'mae':ens_mae,'rmse':np.sqrt(mean_squared_error(true_scores,ens_preds)),
    'r2':ens_r2,'spearman':ens_sp,'acc':ens_acc,
    'pred_scores':ens_preds.tolist(),'true_scores':true_scores,
    'pred_classes':ens_cls,'true_classes':true_cls
}
print(f'🔀 Ensemble         MAE={ens_mae:.2f}  Acc={ens_acc*100:.1f}%')

winner = min(eval_results, key=lambda x: eval_results[x]['mae'])
print(f'\n🏆 Winner: {winner}')

In [ ]:
# ── Summary Table ─────────────────────────────────────
rows = []
for mn, r in eval_results.items():
    rows.append({'Model':mn, 'MAE':round(r['mae'],2), 'RMSE':round(r['rmse'],2),
                 'R²':round(r['r2'],4), 'Spearman':round(r['spearman'],4),
                 'Accuracy':f"{r['acc']*100:.1f}%",
                 '':('🏆' if mn==winner else ('🔀' if mn=='Ensemble' else ''))})
display(pd.DataFrame(rows).style.hide(axis='index').set_caption('📊 Results — All Models')
        .highlight_min(subset=['MAE','RMSE'], color='#DCFCE7')
        .highlight_max(subset=['R²','Spearman'], color='#DCFCE7')
        .set_table_styles([{'selector':'caption','props':[('font-weight','bold'),('font-size','14px')]}]))

In [ ]:
all_names = list(eval_results.keys())   # AraBERT, CAMeL-BERT, Ensemble
clrs_all  = {mn: CLRS.get(mn, '#7C3AED') for mn in all_names}

fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('📊 Evaluation Dashboard', fontsize=14, fontweight='bold')

# 1. MAE + Acc bars
x = np.arange(len(all_names)); w = 0.35
ax = axes[0,0]
b1 = ax.bar(x-w/2, [eval_results[mn]['mae']  for mn in all_names], w,
            label='MAE ↓', color=[clrs_all[mn] for mn in all_names], alpha=0.85)
b2 = ax.bar(x+w/2, [eval_results[mn]['acc']*10 for mn in all_names], w,
            label='Acc×10 ↑', color=[clrs_all[mn] for mn in all_names], alpha=0.45, hatch='//')
ax.set_xticks(x); ax.set_xticklabels(all_names, rotation=10)
ax.set_title('MAE vs Accuracy'); ax.legend()
for bar in list(b1)+list(b2):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+.05,
            f'{bar.get_height():.2f}', ha='center', fontsize=8)

# 2 & 3. Confusion matrices
for ai, mn in enumerate(['AraBERT','CAMeL-BERT']):
    res = eval_results[mn]; ax = axes[0, 1+ai]
    cm  = confusion_matrix(res['true_classes'], res['pred_classes'], labels=CLASS_LABELS)
    pct = cm.astype(float)/cm.sum(axis=1,keepdims=True)*100
    ann = np.array([[f'{v}\n({p:.0f}%)' for v,p in zip(rv,rp)] for rv,rp in zip(cm,pct)])
    sns.heatmap(cm, annot=ann, fmt='', ax=ax, cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                linewidths=.5, linecolor='white')
    ax.set_title(('🏆 ' if mn==winner else '')+f'{mn}\nMAE={res["mae"]:.2f}  Acc={res["acc"]*100:.1f}%')
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')

# 4. Actual vs Predicted (كل الموديلات)
for mn in all_names:
    axes[1,0].scatter(eval_results[mn]['true_scores'], eval_results[mn]['pred_scores'],
                      alpha=.3, s=8, label=mn, color=clrs_all[mn])
axes[1,0].plot([0,100],[0,100],'k--',lw=1.5,label='Perfect')
for t in [T1,T2,T3]:
    axes[1,0].axhline(t,color='gray',linestyle=':',alpha=.4)
    axes[1,0].axvline(t,color='gray',linestyle=':',alpha=.4)
axes[1,0].set_xlabel('Actual'); axes[1,0].set_ylabel('Predicted')
axes[1,0].set_title('Actual vs Predicted'); axes[1,0].legend(fontsize=8)

# 5. F1 per class
x2 = np.arange(len(CLASS_NAMES)); w2 = 0.25
for i, mn in enumerate(all_names):
    f1s = f1_score(eval_results[mn]['true_classes'],eval_results[mn]['pred_classes'],
                   average=None, labels=CLASS_LABELS, zero_division=0)
    bars = axes[1,1].bar(x2+i*w2, f1s, w2, label=mn, color=clrs_all[mn], alpha=0.85)
    for bar,v in zip(bars,f1s):
        axes[1,1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+.01,
                       f'{v:.2f}', ha='center', fontsize=7)
axes[1,1].set_xticks(x2+w2); axes[1,1].set_xticklabels(CLASS_NAMES)
axes[1,1].set_ylim(0,1.15); axes[1,1].set_title('F1 per Class'); axes[1,1].legend(fontsize=8)

# 6. Error distribution
for mn in all_names:
    errs = np.array(eval_results[mn]['pred_scores'])-np.array(eval_results[mn]['true_scores'])
    axes[1,2].hist(errs, bins=30, alpha=.55,
                   label=f'{mn}  μ={errs.mean():.1f}',
                   color=clrs_all[mn], edgecolor='white')
axes[1,2].axvline(0,color='red',linestyle='--',lw=1.5)
axes[1,2].set_xlabel('Error'); axes[1,2].set_title('Error Distribution'); axes[1,2].legend(fontsize=8)

plt.tight_layout(); plt.savefig('04_evaluation.png', dpi=150, bbox_inches='tight'); plt.show()

---
## 8️⃣ Explainability & Inference

In [ ]:
def get_word_importance(text, model, tokenizer, top_k=15):
    model.eval()
    enc = tokenizer(text, return_tensors='pt', truncation=True, max_length=512)
    ids = enc['input_ids'].to(device); msk = enc['attention_mask'].to(device)
    bert_eager = AutoModel.from_pretrained(
        model.bert.config._name_or_path, attn_implementation='eager').to(device)
    bert_eager.load_state_dict(model.bert.state_dict()); bert_eager.eval()
    with torch.no_grad():
        out = bert_eager(input_ids=ids, attention_mask=msk, output_attentions=True)
        if not out.attentions: del bert_eager; return [], []
        cls_att = out.attentions[-1][0][:,0,:].mean(0).cpu().numpy()
    del bert_eager
    tokens = tokenizer.convert_ids_to_tokens(ids[0].cpu().tolist())
    words, scores = [], []
    cw, cs = '', 0
    for tok, s in zip(tokens[1:-1], cls_att[1:-1]):
        if tok.startswith('##'): cw+=tok[2:]; cs=max(cs,s)
        else:
            if cw: words.append(cw); scores.append(cs)
            cw,cs=tok,s
    if cw: words.append(cw); scores.append(cs)
    pairs = sorted(zip(words,scores),key=lambda x:-x[1])[:top_k]
    return zip(*pairs) if pairs else ([],[])

SUGGESTION_TEMPLATES = {
    'ضعيف' : [
        (lambda cv,kws: len(kws)<3,           'مهارات المجال ناقصة → أضف ≥5 مهارات تقنية'),
        (lambda cv,kws: 'الخبرات' not in cv,  'لا يوجد قسم خبرة → أضف خبراتك بالتفصيل'),
        (lambda cv,kws: 'المهارات' not in cv, 'قسم المهارات غائب'),
        (lambda cv,kws: len(cv.split())<150,  'CV قصير → أثرِ المحتوى ليصل 300-500 كلمة'),
        (lambda cv,kws: True,                 'استخدم إنجازات ملموسة بدلاً من المسؤوليات'),
    ],
    'متوسط': [
        (lambda cv,kws: len(kws)<5,                                           'أضف مهارات مجال محددة لرفع الـ ATS'),
        (lambda cv,kws: 'github' not in cv.lower() and 'linkedin' not in cv.lower(), 'أضف رابط GitHub أو LinkedIn'),
        (lambda cv,kws: 'شهادة' not in cv,                                   'أضف شهادات احترافية'),
        (lambda cv,kws: True,                                                 'رتّب خبراتك عكسياً زمنياً'),
        (lambda cv,kws: True,                                                 'أضف أرقاماً لقياس الإنجازات'),
    ],
    'جيد'  : [
        (lambda cv,kws: len(kws)<8,         'زد الكلمات المفتاحية لتحسين ATS'),
        (lambda cv,kws: 'مشروع' not in cv, 'أضف قسم مشاريع'),
        (lambda cv,kws: True,              'خصّص الـ CV لكل وظيفة'),
    ],
    'ممتاز': [
        (lambda cv,kws: True, 'CV ممتاز! راجع التنسيق للتوافق مع ATS'),
        (lambda cv,kws: True, 'أضف قسم توصيات مهنية'),
    ]
}

def analyze_cv(arabic_cv, category, model_name=None, use_ensemble=True):
    cv_clean   = clean_arabic_text(arabic_cv)
    ats_score_val, checks, matched_kws = calculate_ats_score(arabic_cv, category)
    ats_bucket = 'ats_low' if ats_score_val<50 else ('ats_mid' if ats_score_val<75 else 'ats_high')
    input_text = f'{category} {ats_bucket} [SEP] {cv_clean}'

    def _predict_single(mn):
        tok  = tokenizers[mn]; mdl = final_models[mn]
        ds   = CVDatasetSliding(pd.DataFrame([{'input_text':input_text,
                                               'suitability_score':0,'category':category}]),tok)
        item = ds[0]
        mdl.eval()
        with torch.no_grad():
            pred = mdl(item['chunks_ids'].unsqueeze(0).to(device),
                       item['chunks_mask'].unsqueeze(0).to(device),[item['n_chunks']])
        return float(pred.cpu().item())*100

    if use_ensemble:
        score_ara   = _predict_single('AraBERT')
        score_camel = _predict_single('CAMeL-BERT')
        pred_score  = weights['AraBERT']*score_ara + weights['CAMeL-BERT']*score_camel
        used_model  = 'Ensemble'
    else:
        mn         = model_name or winner
        pred_score = _predict_single(mn)
        used_model = mn

    class_name  = CLASS_NAMES[score_to_class(pred_score)]
    domain_kws  = DOMAIN_KEYWORDS.get(category, DEFAULT_KEYWORDS)
    missing_kws = [k for k in domain_kws if k not in matched_kws][:5]
    suggestions = []
    for cond_fn, t in SUGGESTION_TEMPLATES.get(class_name,[]):
        if cond_fn(arabic_cv, matched_kws): suggestions.append(f'📌 {t}')
    for check, passed in checks.items():
        if not passed: suggestions.append(f'⚠️  {ATS_MESSAGES[check]}')

    # Word importance من الـ winner model
    best_mn = winner if winner != 'Ensemble' else 'CAMeL-BERT'
    words, wscores = get_word_importance(input_text, final_models[best_mn],
                                          tokenizers[best_mn], top_k=10)
    return {'model':used_model,'predicted_score':round(pred_score,1),
            'predicted_class':class_name,'thresholds':f'{T1:.0f}/{T2:.0f}/{T3:.0f}',
            'ats_score':ats_score_val,'matched_keywords':matched_kws,
            'missing_keywords':missing_kws,'suggestions':suggestions[:7],
            'top_words':list(zip(words,[round(float(s),4) for s in wscores]))}

# ── Sample ────────────────────────────────────────────
sample = test_df.iloc[2]
result = analyze_cv(sample['arabic_cv'], sample['category'], use_ensemble=True)

out_df = pd.DataFrame([
    ['Model Used',      result['model']],
    ['Category',        sample['category']],
    ['True Score',      f"{sample['suitability_score']:.1f} → {CLASS_NAMES[score_to_class(sample['suitability_score'])]}"],
    ['Predicted Score', f"{result['predicted_score']} → {result['predicted_class']}"],
    ['Thresholds',      result['thresholds']],
    ['ATS Score',       f"{result['ats_score']}/100"],
    ['Matched KWs',     ', '.join(result['matched_keywords']) or 'None'],
    ['Missing KWs',     ', '.join(result['missing_keywords']) or 'None'],
], columns=['Field','Value'])
display(out_df.style.hide(axis='index').set_caption('🔍 CV Analysis Result')
        .set_table_styles([{'selector':'caption','props':[('font-weight','bold'),('font-size','14px')]}]))

print('\n💡 Suggestions:')
for i,s in enumerate(result['suggestions'],1): print(f'  {i}. {s}')
if result['top_words']:
    print('\n🔍 Top Words:')
    for w,s in result['top_words'][:5]: print(f'   {w:<20} {s:.4f}')

---
## 9️⃣ 💾 Save Everything for API

In [ ]:
# Tokenizers
for mn, tok in tokenizers.items():
    p = SAVE_DIR/f'{mn.replace("-","_")}_tokenizer'
    tok.save_pretrained(p); print(f'✅ Tokenizer → {p}')

# Config
config = {
    'thresholds'      : {'T1':T1,'T2':T2,'T3':T3},
    'class_names'     : CLASS_NAMES,'class_labels':CLASS_LABELS,
    'winner_model'    : winner,'models':MODELS,
    'hyperparams'     : {**BEST_PARAMS,'lr':str(BEST_PARAMS['lr']),'head_lr':str(BEST_PARAMS['head_lr'])},
    'ensemble_weights': {mn:round(w,4) for mn,w in weights.items()},
    'eval_summary'    : {mn:{'mae':round(r['mae'],4),'rmse':round(r['rmse'],4),
                              'r2':round(r['r2'],4),'spearman':round(r['spearman'],4),
                              'acc':round(r['acc'],4)} for mn,r in eval_results.items()}
}
with open(SAVE_DIR/'config.json','w',encoding='utf-8') as f:
    json.dump(config,f,ensure_ascii=False,indent=2)
print('✅ config.json')

with open(SAVE_DIR/'ats_config.json','w',encoding='utf-8') as f:
    json.dump({'domain_keywords':DOMAIN_KEYWORDS,'default_keywords':DEFAULT_KEYWORDS,
               'ats_messages':ATS_MESSAGES},f,ensure_ascii=False,indent=2)
print('✅ ats_config.json')

with open(SAVE_DIR/'eval_results.pkl','wb') as f:
    pickle.dump(eval_results,f)
print('✅ eval_results.pkl')

model_code = '''
"""Model classes — import in your API"""
import torch, torch.nn as nn, torch.nn.functional as F
from transformers import AutoModel

class AttentionPooling(nn.Module):
    def __init__(self, hidden_size=768):
        super().__init__()
        self.attention=nn.Sequential(nn.Linear(hidden_size,256),nn.Tanh(),nn.Linear(256,1))
    def forward(self, cls_embeddings, n_chunks):
        pooled=[]
        for i in range(cls_embeddings.shape[0]):
            n=n_chunks[i]; emb=cls_embeddings[i,:n,:]
            att=F.softmax(self.attention(emb),dim=0)
            pooled.append((att*emb).sum(dim=0))
        return torch.stack(pooled)

class CVRegressor(nn.Module):
    def __init__(self, model_path, dropout=0.2):
        super().__init__()
        self.bert     = AutoModel.from_pretrained(model_path)
        self.att_pool = AttentionPooling(768)
        self.head     = nn.Sequential(
            nn.Linear(768,256),nn.GELU(),nn.Dropout(dropout),
            nn.Linear(256,64), nn.GELU(),nn.Dropout(dropout),
            nn.Linear(64,1),   nn.Sigmoid())
    def forward(self, chunks_ids, chunks_mask, n_chunks):
        B,C,L=chunks_ids.shape
        out=self.bert(input_ids=chunks_ids.view(B*C,L),
                      attention_mask=chunks_mask.view(B*C,L),output_attentions=False)
        cls=out.last_hidden_state[:,0,:].view(B,C,768)
        return self.head(self.att_pool(cls,n_chunks)).squeeze(-1)

class HuberMSELoss(nn.Module):
    def __init__(self, delta=0.1, alpha=0.7):
        super().__init__()
        self.mse=nn.MSELoss(); self.huber=nn.HuberLoss(delta=delta); self.alpha=alpha
    def forward(self, pred, target):
        return self.alpha*self.mse(pred,target)+(1-self.alpha)*self.huber(pred,target)
'''
with open(SAVE_DIR/'model_classes.py','w',encoding='utf-8') as f:
    f.write(model_code)
print('✅ model_classes.py')

# File listing
rows=[]
for p in sorted(SAVE_DIR.rglob('*')):
    if p.is_file():
        kb=p.stat().st_size/1024
        rows.append({'File':str(p.relative_to(SAVE_DIR)),
                     'Size':f'{kb/1024:.1f} MB' if kb>1024 else f'{kb:.0f} KB'})
display(pd.DataFrame(rows).style.hide(axis='index').set_caption('📦 saved_model/')
        .set_table_styles([{'selector':'caption','props':[('font-weight','bold')]}]))

print(f'\n🎉 Done!  Winner:{winner}  '
      f'MAE={eval_results[winner]["mae"]:.2f}  '
      f'Acc={eval_results[winner]["acc"]*100:.1f}%')